# Few-shot 学习 (Few-shot Learning)

> **学习目标**：掌握 Few-shot 提示技术，通过示例引导模型输出

---

## 目录

1. [Few-shot 概述](#1-few-shot-概述)
2. [示例设计原则](#2-示例设计原则)
3. [示例选择策略](#3-示例选择策略)
4. [Few-shot 实现](#4-few-shot-实现)
5. [实战案例](#5-实战案例)

In [ ]:
import sys
sys.path.append('..')

from src.few_shot import (
    Example, FewShotPrompt, FewShotTemplates,
    RandomExampleSelector, SemanticExampleSelector, DiversityExampleSelector
)
import numpy as np

## 1. Few-shot 概述

**Few-shot Learning**：通过提供少量示例，让模型学习任务模式。

```
示例1: 输入 → 输出
示例2: 输入 → 输出
示例3: 输入 → 输出
新输入 → ?
```

In [ ]:
# Few-shot vs Zero-shot 对比

# Zero-shot
zero_shot = """将情感分类为正面或负面：

文本：今天的天气真好，心情愉快。
情感："""

# Few-shot
few_shot = """将情感分类为正面或负面：

文本：这部电影太精彩了！
情感：正面

文本：服务态度很差，不会再来了。
情感：负面

文本：今天的天气真好，心情愉快。
情感："""

print("=== Zero-shot ===")
print(zero_shot)
print("\n=== Few-shot ===")
print(few_shot)

## 2. 示例设计原则

### 2.1 多样性原则

In [ ]:
# 好的示例集：覆盖不同类别
good_examples = [
    Example({"text": "太棒了，强烈推荐！"}, "正面"),
    Example({"text": "很失望，质量太差"}, "负面"),
    Example({"text": "还可以，一般般"}, "中性"),
]

# 差的示例集：类别不均衡
bad_examples = [
    Example({"text": "很好"}, "正面"),
    Example({"text": "不错"}, "正面"),
    Example({"text": "挺好"}, "正面"),
]

print("好的示例集覆盖了所有类别")
print("差的示例集只有正面类别，会导致模型偏向")

In [ ]:
# 2.2 示例数量选择

example_count_guide = """
| 示例数 | 效果   | 适用场景     |
|--------|--------|-------------|
| 1-2    | 基础   | 简单任务     |
| 3-5    | 良好   | 大多数任务   |
| 5-10   | 优秀   | 复杂任务     |
| >10    | 边际递减 | 特殊情况   |
"""
print(example_count_guide)

## 3. 示例选择策略

In [ ]:
# 创建示例库
example_pool = [
    Example({"input": "我很开心"}, "positive"),
    Example({"input": "我很难过"}, "negative"),
    Example({"input": "今天天气不错"}, "neutral"),
    Example({"input": "太棒了"}, "positive"),
    Example({"input": "真糟糕"}, "negative"),
    Example({"input": "还行吧"}, "neutral"),
]

# 3.1 随机选择
random_selector = RandomExampleSelector(examples=example_pool, seed=42)
selected = random_selector.select("我感到很快乐", k=3)
print("=== 随机选择 ===")
for ex in selected:
    print(f"  {ex.input_data['input']} -> {ex.output}")

In [ ]:
# 3.2 语义相似度选择
semantic_selector = SemanticExampleSelector(examples=example_pool)
selected = semantic_selector.select("我感到很快乐", k=3)
print("=== 语义相似度选择 ===")
for ex in selected:
    print(f"  {ex.input_data['input']} -> {ex.output}")

In [ ]:
# 3.3 多样性选择 (MMR)
diversity_selector = DiversityExampleSelector(examples=example_pool, lambda_param=0.5)
selected = diversity_selector.select("我感到很快乐", k=3)
print("=== 多样性选择 (MMR) ===")
for ex in selected:
    print(f"  {ex.input_data['input']} -> {ex.output}")

## 4. Few-shot 实现

In [ ]:
# 使用 FewShotPrompt 构建提示

sentiment_prompt = FewShotPrompt(
    examples=[
        Example({"text": "这个产品太棒了！"}, "正面"),
        Example({"text": "服务态度很差"}, "负面"),
        Example({"text": "还可以吧"}, "中性"),
    ],
    example_template="文本：{text}\n情感：{output}",
    prefix="判断以下文本的情感倾向（正面/负面/中性）：\n\n",
    suffix="\n\n文本：{text}\n情感：",
    input_variables=["text"]
)

prompt = sentiment_prompt.format(text="这家餐厅的菜品非常美味")
print(prompt)

In [ ]:
# 使用预定义模板

print("=== 情感分类 ===")
sentiment = FewShotTemplates.sentiment_classification()
print(sentiment.format(text="这个手机拍照效果很好"))

print("\n=== 翻译 ===")
translation = FewShotTemplates.translation()
print(translation.format(source="早上好"))

## 5. 实战案例

In [ ]:
# 案例1：命名实体识别

ner_prompt = FewShotPrompt(
    examples=[
        Example(
            {"text": "马云创立了阿里巴巴公司。"},
            "人物：马云\n组织：阿里巴巴公司"
        ),
        Example(
            {"text": "北京是中国的首都。"},
            "地点：北京、中国"
        ),
    ],
    example_template="文本：{text}\n实体：\n{output}",
    prefix="从文本中提取命名实体（人物、地点、组织）：\n\n",
    suffix="\n\n文本：{text}\n实体：\n",
    input_variables=["text"]
)

print(ner_prompt.format(text="张三在上海的腾讯公司工作。"))

In [ ]:
# 案例2：SQL生成

sql_prompt = FewShotPrompt(
    examples=[
        Example(
            {"question": "查询所有用户"},
            "SELECT * FROM users;"
        ),
        Example(
            {"question": "查询年龄大于18的用户"},
            "SELECT * FROM users WHERE age > 18;"
        ),
        Example(
            {"question": "统计每个城市的用户数量"},
            "SELECT city, COUNT(*) FROM users GROUP BY city;"
        ),
    ],
    example_template="问题：{question}\nSQL：{output}",
    prefix="将自然语言转换为SQL查询：\n\n",
    suffix="\n\n问题：{question}\nSQL：",
    input_variables=["question"]
)

print(sql_prompt.format(question="查询订单金额大于100的所有订单"))

## 总结

1. **Few-shot** 通过示例引导模型学习任务模式
2. **示例设计**：多样性、代表性、格式一致性
3. **选择策略**：随机、语义相似、多样性平衡
4. **最佳实践**：3-5个示例通常足够

下一节：**Chain-of-Thought** 思维链推理